# AO3 Tag Cluster Network Resume

Renders `ao3_tag_cluster_network.html` from an **already-computed**
`ao3_tag_clusters.csv` (produced by `ao3_tag_analysis.py`/`ao3_tag_analysis.ipynb`),
without re-running community detection (`detect_communities`) or cluster-size
merging (`merge_small_communities`).

Use this when you already have `ao3_tag_clusters.csv` from a prior run and
just want to regenerate the network graph -- e.g. if you tweaked filtering
or just want the HTML again without re-driving the whole pipeline from the
CSV/incidence-matrix step. Skips straight to loading cluster membership and
rendering.

Requires `ao3_tag_analysis.py` and `ao3_tag_visualizer.py` (imported directly,
unlike the other notebooks in this repo, which hand-copy function source --
this one is a thin resume utility built on top of them, not a standalone tool)
in the same directory.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present.
# Safe to re-run -- already-satisfied requirements are a fast no-op. You've
# almost certainly already installed these if you've run ao3_tag_analysis.py.
%pip install -q pandas numpy scipy networkx pyvis

In [ ]:
import pandas as pd

import ao3_tag_analysis as analysis
import ao3_tag_visualizer as viz

## Configuration

In [ ]:
INPUT = "ao3_tag_metadata.csv"

# Must match the flags used for the ao3_tag_analysis.py run that produced
# CLUSTERS_CSV below -- these determine which tags/edges go into the graph,
# and need to line up with what's already in the CSV.
ALL_TAGS = True
TOP_TAGS = 60          # ignored if ALL_TAGS is True
MIN_PAIR_COUNT = 2

CLUSTERS_CSV = "ao3_tag_clusters.csv"
NETWORK_OUT = "ao3_tag_cluster_network.html"

## Rebuild the graph and render the network

`build_all_fields_pair_data` + `build_cluster_graph` are the fast steps
(seconds, even at `--all-tags` scale) -- this loads cluster membership
directly from `CLUSTERS_CSV` instead of calling `detect_communities`/
`merge_small_communities`, which is what skips the slow part.

In [ ]:
df = viz.load_metadata(INPUT)
top_tags = None if ALL_TAGS else TOP_TAGS
pair_stats, keep_tags = analysis.build_all_fields_pair_data(df, top_tags, MIN_PAIR_COUNT)
print(f"{len(keep_tags)} tags kept, {len(pair_stats)} surviving pairs")

cluster_graph = analysis.build_cluster_graph(pair_stats, keep_tags)
clusters_df = pd.read_csv(CLUSTERS_CSV)
print(f"loaded {len(clusters_df)} tags, {clusters_df['cluster_id'].nunique()} clusters from {CLUSTERS_CSV}")

analysis.color_graph_by_cluster(cluster_graph, clusters_df)
net = viz.render_network(cluster_graph, NETWORK_OUT, notebook=True,
                          inject_filters=analysis._inject_cluster_filter_controls)
iframe = net.show(NETWORK_OUT, notebook=True)
# net.show() internally rewrites the file from pyvis's raw template, wiping
# out render_network's post-processing -- redo it on the file it just wrote,
# same dance as ao3_tag_analysis.ipynb's own cluster-network cell.
viz._strip_bootstrap_cdn(NETWORK_OUT)
analysis._inject_cluster_filter_controls(NETWORK_OUT, cluster_graph)
viz._inject_stabilize_then_stop(NETWORK_OUT)
iframe

## Done

`{NETWORK_OUT}` is now in the notebook's working directory. Re-run from the
Configuration cell with different `CLUSTERS_CSV`/`NETWORK_OUT` values to
resume a different run.